# 02 — Le attribuzioni dei due modelli (tentativo 13, fase 2)

Per ogni articolo del campione, con il modello (TF-IDF e ModernBERT) **del fold in cui l'articolo era nel test**:
Integrated Gradients del punteggio di affidabilità f = log P(l3..l5) − log P(l1..l2), baseline "testo vuoto"
(`comune_xai.ig_feature`, `ig_modernbert`; 50 passi per il TF-IDF, 64 per ModernBERT, in precisione piena). Per
ModernBERT si salva anche **gradiente × input** (`attr_modernbert_gxi`), un secondo metodo usato come controllo di
robustezza delle quote per categoria (per ModernBERT la completezza di IG non è esatta: si riporta l'errore). Le attribuzioni (sui token
del vettorizzatore e sui sottotoken di ModernBERT) si riportano sui token spaCy di `01` in proporzione ai caratteri.

ModernBERT legge al più 1024 token: `in_finestra` dice se un token sta nella parte letta. Il TF-IDF legge tutto
l'articolo. Le analisi principali usano la parte letta da tutti e due.

Controlli: per ogni articolo la somma delle attribuzioni deve valere f(x) − f(baseline) (completezza di IG); si salva
la differenza.

Uscite: `token_attribuzioni.csv.gz` (i token di `01` con `attr_feature`, `attr_modernbert`, `in_finestra`) e
`articoli_attribuzioni.csv` (f(x), f(base), completezza, previsione dei due modelli).

In [1]:
PROVA = False      # True: i file di prova di 01

In [2]:
import time

import joblib
import numpy as np
import pandas as pd
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

from comune_xai import MODELLI, RESULTS, VERSIONI, ig_feature, ig_modernbert, su_token

OUT = RESULTS / ("prova" if PROVA else "")
camp = pd.read_csv(OUT / "campione.csv")
tok = pd.read_csv(OUT / "token_campione.csv.gz")
art = pd.read_csv(VERSIONI / "dataset.csv", usecols=["article_id", "text"]).set_index("article_id")
testi = art.text.fillna("").reindex(camp.article_id)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
while torch.cuda.is_available() and (torch.cuda.mem_get_info()[0] + torch.cuda.memory_reserved()) / 2**30 < 6.0:
    print("GPU occupata da altri: riprovo tra 5 minuti", flush=True); time.sleep(300)
print(f"{len(camp)} articoli, {len(tok)} token; fold {sorted(camp.fold.unique())}; {device}")

/home/fabio.pinelli/John/johnVenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


300 articoli, 318898 token; fold [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]; cuda


/home/fabio.pinelli/John/johnVenv/lib/python3.12/site-packages/torch/cuda/__init__.py:716: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


In [3]:
tok["attr_feature"] = 0.0; tok["attr_modernbert"] = 0.0; tok["attr_modernbert_gxi"] = 0.0; tok["in_finestra"] = False
per_art = tok.groupby("article_id").indices
righe = []
t0 = time.time()
for f in sorted(camp.fold.unique()):
    mf = joblib.load(MODELLI / "feature" / f"fold{f}.joblib")
    d = MODELLI / "modernbert" / f"fold{f}"
    tk = AutoTokenizer.from_pretrained(d)
    mb = AutoModelForSequenceClassification.from_pretrained(d, reference_compile=False).to(device).eval()
    for aid in camp.article_id[camp.fold == f]:
        testo = testi[aid]; ix = per_art[aid]
        inizi, fini = tok.inizio.to_numpy()[ix], tok.fine.to_numpy()[ix]
        pf, fx_f, f0_f = ig_feature(mf, testo)
        pm, fx_m, f0_m, delta, fine_finestra = ig_modernbert(mb, tk, testo, device)
        af = su_token(inizi, fini, pf)
        am = su_token(inizi, fini, [(s, e, a) for s, e, a, _ in pm]); ag = su_token(inizi, fini, [(s, e, g) for s, e, _, g in pm])
        tok.loc[tok.index[ix], "attr_feature"] = af
        tok.loc[tok.index[ix], "attr_modernbert"] = am
        tok.loc[tok.index[ix], "attr_modernbert_gxi"] = ag
        tok.loc[tok.index[ix], "in_finestra"] = fini <= fine_finestra
        righe.append({"article_id": aid, "fold": f, "f_feature": fx_f, "f0_feature": f0_f, "completezza_feature": af.sum() - (fx_f - f0_f),
                      "f_modernbert": fx_m, "f0_modernbert": f0_m, "delta_ig_modernbert": delta, "completezza_modernbert": am.sum() - (fx_m - f0_m),
                      "fine_finestra": fine_finestra, "caratteri": len(testo)})
    del mb; torch.cuda.empty_cache()
    print(f"fold {f}: fatto ({(time.time() - t0) / 60:.1f} min)", flush=True)
ar = pd.DataFrame(righe)
ar.to_csv(OUT / "articoli_attribuzioni.csv", index=False)
tok.to_csv(OUT / "token_attribuzioni.csv.gz", index=False)
print("completezza (somma delle attribuzioni − [f(x) − f(base)]), mediana in valore assoluto:",
      f"feature {ar.completezza_feature.abs().median():.4f}, ModernBERT {ar.completezza_modernbert.abs().median():.4f}")
print(f"correlazione tra i punteggi f(x) dei due modelli sugli articoli: {np.corrcoef(ar.f_feature, ar.f_modernbert)[0, 1]:.3f}")
ar.describe().round(3)

/home/fabio.pinelli/John/johnVenv/lib/python3.12/site-packages/torch/cuda/__init__.py:716: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


fold 1: fatto (9.4 min)


fold 2: fatto (18.1 min)


fold 3: fatto (27.4 min)


fold 4: fatto (35.5 min)


fold 5: fatto (42.2 min)


completezza (somma delle attribuzioni − [f(x) − f(base)]), mediana in valore assoluto: feature 0.0000, ModernBERT 0.6445
correlazione tra i punteggi f(x) dei due modelli sugli articoli: 0.477


,article_id,fold,f_feature,f0_feature,completezza_feature,f_modernbert,f0_modernbert,delta_ig_modernbert,completezza_modernbert,fine_finestra,caratteri
count,300.000,300.000,300.000,300.000,300.0,300.000,300.000,300.000,300.000,300.000,300.000
mean,90922.627,2.953,0.575,-0.385,0.0,0.630,0.865,-0.013,-0.054,3519.523,5623.527
std,49468.318,1.413,2.138,0.164,0.0,1.134,2.030,1.518,1.450,1443.175,6668.370
min,16.000,1.000,-5.837,-0.676,-0.0,-1.953,-4.197,-13.297,-11.805,621.000,621.000
25%,52244.000,2.000,-0.558,-0.419,-0.0,-0.105,0.957,-0.575,-0.633,2305.500,2305.500
50%,91594.500,3.000,0.439,-0.360,-0.0,0.455,1.220,0.087,-0.035,4045.000,4086.000
75%,127627.750,4.000,1.697,-0.280,0.0,1.379,2.510,0.738,0.644,4714.000,6574.750
max,189045.000,5.000,8.251,-0.197,0.0,4.058,3.126,7.670,7.258,5805.000,74414.000
